# Quarantine Reprocess Batch

Reprocess resolved quarantine records back through the appropriate pipeline stage.

**Purpose**: Extract fixed records from quarantine and route them back to the correct pipeline stage for reprocessing

**Workflow**:
1. Identify RESOLVED records by reprocess_batch_id
2. Parse payload_json back into structured data
3. Route records to appropriate stage (bronze/silver/semantic/warehouse)
4. Update quarantine status upon successful reprocessing
5. Handle reprocessing failures

**Integration Points**:
- Bronze layer: Re-ingest raw data
- Silver layer: Reapply transformations
- Semantic layer: Rebuild business entities
- Warehouse layer: Reload facts/dimensions

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, from_json, schema_of_json, current_timestamp, lit
)
from pyspark.sql.types import StructType
import json
from datetime import datetime

# Configuration
QUARANTINE_TABLE = "workspace.quarantine.quarantine_jobs"

# Target tables by stage (customize based on your architecture)
STAGE_TARGETS = {
    "BRONZE": "workspace.bronze.{source_name}_raw",
    "SILVER": "workspace.silver.{source_name}_clean",
    "SEMANTIC": "workspace.semantic.{entity_name}",
    "WAREHOUSE": "workspace.warehouse.{table_name}"
}

# Status constants
STATUS_RESOLVED = "RESOLVED"
STATUS_REPROCESSED = "REPROCESSED"
STATUS_REPROCESS_FAILED = "REPROCESS_FAILED"

In [0]:
def get_resolved_records(
    reprocess_batch_id: str = None,
    source_name: str = None,
    failure_stage: str = None,
    limit: int = 1000
) -> DataFrame:
    """
    Get RESOLVED records ready for reprocessing.
    
    Args:
        reprocess_batch_id: Specific batch to reprocess
        source_name: Filter by source
        failure_stage: Filter by original failure stage
        limit: Max records to reprocess
        
    Returns:
        DataFrame of resolved quarantine records
    """
    df = spark.table(QUARANTINE_TABLE).filter(
        col("reprocess_status") == STATUS_RESOLVED
    )
    
    if reprocess_batch_id:
        df = df.filter(col("reprocess_batch_id") == reprocess_batch_id)
    if source_name:
        df = df.filter(col("source_name") == source_name)
    if failure_stage:
        df = df.filter(col("failure_stage") == failure_stage)
    
    return df.limit(limit)


def display_reprocess_candidates():
    """Display summary of records ready for reprocessing."""
    df = spark.table(QUARANTINE_TABLE).filter(
        col("reprocess_status") == STATUS_RESOLVED
    )
    
    print("=" * 80)
    print("REPROCESS CANDIDATES")
    print("=" * 80)
    
    total = df.count()
    print(f"\n✅ Total RESOLVED records: {total}\n")
    
    if total > 0:
        print("By Reprocess Batch ID:")
        df.groupBy("reprocess_batch_id").count() \
            .orderBy("count", ascending=False).show()
        
        print("\nBy Source and Stage:")
        df.groupBy("source_name", "failure_stage").count() \
            .orderBy("count", ascending=False).show()

# Display candidates
display_reprocess_candidates()

In [0]:
def parse_quarantine_payloads(quarantine_df: DataFrame) -> DataFrame:
    """
    Parse JSON payloads back into structured DataFrame.
    
    Args:
        quarantine_df: DataFrame from quarantine table
        
    Returns:
        DataFrame with original record structure
    """
    if quarantine_df.isEmpty():
        return None
    
    # Get sample payload to infer schema
    sample_payload = quarantine_df.select("payload_json").first()[0]
    
    # Infer schema from sample
    schema = schema_of_json(lit(sample_payload))
    
    # Parse all payloads with inferred schema
    parsed_df = quarantine_df.withColumn(
        "parsed_payload",
        from_json(col("payload_json"), schema)
    )
    
    # Extract all fields from parsed payload
    parsed_df = parsed_df.select(
        col("quarantine_id"),
        col("source_name"),
        col("failure_stage"),
        col("reprocess_batch_id"),
        col("parsed_payload.*")
    )
    
    return parsed_df

In [0]:
def route_to_target_stage(
    records_df: DataFrame,
    failure_stage: str,
    source_name: str
) -> bool:
    """
    Route reprocessed records to appropriate target table.
    
    Args:
        records_df: DataFrame of parsed records
        failure_stage: Original failure stage
        source_name: Source system name
        
    Returns:
        True if successful, False otherwise
    """
    try:
        # Determine target table based on stage
        if failure_stage == "BRONZE":
            target_table = f"workspace.bronze.{source_name}_reprocessed"
            # For bronze, you might want to merge back into raw table
        elif failure_stage == "SILVER":
            target_table = f"workspace.silver.{source_name}_reprocessed"
        elif failure_stage == "SEMANTIC":
            target_table = f"workspace.semantic.{source_name}_reprocessed"
        elif failure_stage == "WAREHOUSE":
            target_table = f"workspace.warehouse.{source_name}_reprocessed"
        else:
            print(f"⚠️  Unknown failure stage: {failure_stage}")
            return False
        
        # Add reprocessing metadata
        output_df = records_df.withColumn(
            "reprocessed_at", current_timestamp()
        ).withColumn(
            "is_reprocessed", lit(True)
        )
        
        # Write to target (append mode)
        output_df.write.mode("append").saveAsTable(target_table)
        
        count = output_df.count()
        print(f"✓ Routed {count} records to {target_table}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error routing to {failure_stage}: {str(e)}")
        return False

In [0]:
def update_reprocessed_status(
    quarantine_ids: list,
    success: bool
) -> int:
    """
    Update quarantine records after reprocessing attempt.
    
    Args:
        quarantine_ids: List of quarantine IDs
        success: True if reprocessing succeeded, False if failed
        
    Returns:
        Count of records updated
    """
    if not quarantine_ids:
        return 0
    
    new_status = STATUS_REPROCESSED if success else STATUS_REPROCESS_FAILED
    ids_str = "', '".join(quarantine_ids)
    
    update_sql = f"""
    UPDATE {QUARANTINE_TABLE}
    SET reprocess_status = '{new_status}',
        resolved_at = current_timestamp()
    WHERE quarantine_id IN ('{ids_str}')
    """
    
    spark.sql(update_sql)
    
    status_emoji = "✅" if success else "❌"
    print(f"{status_emoji} Updated {len(quarantine_ids)} records to: {new_status}")
    
    return len(quarantine_ids)

In [0]:
def reprocess_batch(
    reprocess_batch_id: str,
    source_name: str = None,
    failure_stage: str = None,
    dry_run: bool = False
) -> dict:
    """
    Main function to reprocess a batch of resolved records.
    
    Args:
        reprocess_batch_id: Batch ID to reprocess
        source_name: Optional source filter
        failure_stage: Optional stage filter
        dry_run: If True, only display what would be reprocessed
        
    Returns:
        Dictionary with reprocessing results
    """
    results = {
        "batch_id": reprocess_batch_id,
        "total_records": 0,
        "successful": 0,
        "failed": 0
    }
    
    print("=" * 80)
    print(f"REPROCESSING BATCH: {reprocess_batch_id}")
    print("=" * 80)
    
    # Get resolved records
    resolved_df = get_resolved_records(
        reprocess_batch_id=reprocess_batch_id,
        source_name=source_name,
        failure_stage=failure_stage
    )
    
    total = resolved_df.count()
    results["total_records"] = total
    
    if total == 0:
        print("\nℹ️  No records found to reprocess")
        return results
    
    print(f"\n📋 Found {total} records to reprocess\n")
    
    if dry_run:
        print("🔍 DRY RUN - No actual reprocessing\n")
        resolved_df.groupBy("source_name", "failure_stage").count().show()
        return results
    
    # Group by source and stage
    groups = resolved_df.select("source_name", "failure_stage").distinct().collect()
    
    for group in groups:
        src_name = group["source_name"]
        stage = group["failure_stage"]
        
        print(f"\n🔄 Processing: {src_name} / {stage}")
        
        # Get records for this group
        group_df = resolved_df.filter(
            (col("source_name") == src_name) &
            (col("failure_stage") == stage)
        )
        
        # Parse payloads
        parsed_df = parse_quarantine_payloads(group_df)
        
        if parsed_df is None or parsed_df.isEmpty():
            print(f"  ⚠️  No records to parse for {src_name}")
            continue
        
        # Route to target stage
        success = route_to_target_stage(parsed_df, stage, src_name)
        
        # Update quarantine status
        qids = [row["quarantine_id"] for row in group_df.collect()]
        update_reprocessed_status(qids, success)
        
        if success:
            results["successful"] += len(qids)
        else:
            results["failed"] += len(qids)
    
    print("\n" + "=" * 80)
    print("REPROCESSING COMPLETE")
    print(f"  ✅ Successful: {results['successful']}")
    print(f"  ❌ Failed: {results['failed']}")
    print("=" * 80)
    
    return results

In [0]:
# Example usage

# First, do a dry run to preview
print("\n🔍 DRY RUN - Preview reprocessing candidates\n")
dry_results = reprocess_batch(
    reprocess_batch_id="REPROCESS_20240601_001",
    dry_run=True
)

# Uncomment to actually reprocess
# results = reprocess_batch(
#     reprocess_batch_id="REPROCESS_20240601_001",
#     dry_run=False
# )
# 
# print(f"\nReprocessing Results: {results}")

In [0]:
def reprocess_all_resolved(batch_size: int = 100) -> dict:
    """
    Reprocess all RESOLVED records in batches.
    
    Args:
        batch_size: Number of records per batch
        
    Returns:
        Summary of all reprocessing
    """
    summary = {"total_batches": 0, "total_records": 0, "successful": 0, "failed": 0}
    
    # Get unique reprocess_batch_ids
    resolved_df = spark.table(QUARANTINE_TABLE).filter(
        col("reprocess_status") == STATUS_RESOLVED
    )
    
    batch_ids = [row["reprocess_batch_id"] 
                 for row in resolved_df.select("reprocess_batch_id").distinct().collect()]
    
    print(f"\n📊 Found {len(batch_ids)} unique reprocess batches\n")
    
    for batch_id in batch_ids:
        print(f"\n{'=' * 80}")
        print(f"Processing batch: {batch_id}")
        print(f"{'=' * 80}")
        
        results = reprocess_batch(batch_id, dry_run=False)
        
        summary["total_batches"] += 1
        summary["total_records"] += results["total_records"]
        summary["successful"] += results["successful"]
        summary["failed"] += results["failed"]
    
    print("\n" + "=" * 80)
    print("ALL BATCHES COMPLETE")
    print(f"  📊 Batches processed: {summary['total_batches']}")
    print(f"  📋 Total records: {summary['total_records']}")
    print(f"  ✅ Successful: {summary['successful']}")
    print(f"  ❌ Failed: {summary['failed']}")
    print("=" * 80)
    
    return summary

# Uncomment to reprocess all
# all_results = reprocess_all_resolved()